---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

## Parameters

In [1]:
knitr::opts_chunk$set(echo = TRUE)

to_read <- FALSE
to_split <- TRUE
to_sample <- FALSE
to_write <- TRUE
to_group <- TRUE
to_filter <- FALSE # unused
to_profvis <- FALSE
to_chunk <- TRUE # doesn't work if false; not chunking is deprecated.
to_view_checks <- TRUE
to_view_checks_parallelized <- FALSE
to_parallelize <- TRUE

year_to_load <- "2018"
version <- "v2"

split_chunks <- 5

# which of the split chunks to process;
# write NA if it should process everything in a loop
split_chunk_to_process <- NA

if (is.na(split_chunk_to_process)) {
  part <- NULL
} else {
  part <- split_chunk_to_process
}

seed <- 123
rows_to_show <- 10

drop_cols <- c(
  paste0("ICDCODE", 13:14),
  "ICCODED15",
  paste0("ICDCODE", 16:170)
)

icd_cols <- paste0("clin_icd", 1:12)
rvs_cols <- paste0("clin_rvs", 1:20)

set.seed(seed)
options(future.globals.maxSize = 1024 * 1024^2)

global_seed <- seed # for parallelized operations


## Load Required Libraries & Initial Functions

In [2]:
options(verbose = FALSE)
options(warn = -1)


In [3]:
library(here)
source(here("data-cleaning", "r_scripts", "libraries.R"))
source(here("data-cleaning", "r_scripts", "data-formats.R"))
source(here("data-cleaning", "r_scripts", "file-paths.R"))
# source(here("data-cleaning", "r_scripts", "everything.R"))


here() starts at C:/Users/resur/Documents/drg-pipeline



In [4]:
options(warn = 1)


### Read Total Rows & Sampling Parameters

In [5]:
if (file.exists(total_rows_file(part))) {
  total_rows <- readRDS(total_rows_file(part))
} else {
  total_rows <- fread(full_claims_file(part), select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file(part))
}

sample_size_divisor <- 5
if (to_split) {
  sample_size <- ceiling(total_rows / split_chunks / sample_size_divisor)
} else {
  sample_size <- ceiling(total_rows / sample_size_divisor)
}


## Source Functions

In [6]:
source(here("data-cleaning", "r_scripts", "general-functions.R"))
# source(here("data-cleaning", "r_scripts", "clean-data-mini-functions.R"))
# source(here("data-cleaning", "r_scripts", "profvis.R"))
source(here("data-cleaning", "r_scripts", "main-functions.R"))
source(here("data-cleaning", "r_scripts", "io-functions.R"))
source(here("data-cleaning", "r_scripts", "icd-functions.R"))
source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("data-cleaning", "r_scripts", "grouper-functions.R"))


## Load Mapping Data

In [7]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]
# head(proc)

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]
# head(rvs_icd9)

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))
# head(acr_rvs)

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")
# head(tdrg_icd10)

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read, Process, Export Data (Looping through all parts)

In [8]:
all_parts_summaries <- list()
all_parts_statistics <- list()

# Suppress interim output
suppress_interim_output <- function(expr) {
  suppressMessages(suppressWarnings(capture.output(expr, file = NULL)))
}

# Main block modified to include the updated read_and_save_partial function
if (is.na(split_chunk_to_process)) {
  if (to_split) {
    rows_per_part <- ceiling(total_rows / split_chunks)
    for (part in 1:split_chunks) {
      chunk_file <- if (to_sample) {
        sampled_claims_file(part)
      } else {
        full_claims_file(part)
      }

      if (!file.exists(chunk_file)) {
        start_row <- (part - 1) * rows_per_part + 1
        end_row <- min(part * rows_per_part, total_rows)
        read_and_save_partial(start_row, end_row, part)
      } else {
        # print(paste("File already exists:", chunk_file))
      }
    }
  }

  tic("Total execution time:")

  for (part in 1:split_chunks) {
    chunk_file <- if (to_sample) {
      sampled_claims_file(part)
    } else {
      full_claims_file(part)
    }

    if (!file.exists(chunk_file)) {
      print(paste("File does not exist. Creating file:", chunk_file))
      rows_per_part <- ceiling(total_rows / split_chunks)
      start_row <- (part - 1) * rows_per_part + 1
      end_row <- min(part * rows_per_part, total_rows)
      read_and_save_partial(start_row, end_row, part)
    }

    dt <- main_read_function(file = chunk_file)
    # print(paste("Number of rows after reading:", nrow(dt))) # Add this line
    if (to_profvis) {
      p <- profvis({
        if (to_chunk) {
          num_cores <- max(1, availableCores() - 1)

          suppress_interim_output({
            result <- parallelize_and_summarize(
              dt, num_cores,
              to_view_checks = to_view_checks_parallelized, global_seed,
              rows_to_show, rvs_icd9, tdrg_icd10, acc_pdx, to_parallelize
            )
          })

          dt <- result$dt
          all_parts_summaries[[part]] <- result$consolidated_summary
          all_parts_statistics[[part]] <- result$aggregate_statistics

          # Print summary tables for individual parts
          # print_summary_tables(result, rows_to_show)
        } else {
          stop("Error: to_chunk must be TRUE; not chunking is deprecated.")
        }
      })
      htmlwidgets::saveWidget(
        p,
        file = here(
          "git-ignored-files", "profvis",
          paste0(
            "parallelized_part_",
            paste0(part, "_of_", split_chunks), "_.html"
          )
        ),
        selfcontained = TRUE
      )
    } else {
      if (to_chunk) {
        num_cores <- max(1, availableCores() - 1)

        suppress_interim_output({
          result <- parallelize_and_summarize(
            dt, num_cores,
            to_view_checks = to_view_checks_parallelized, global_seed,
            rows_to_show, rvs_icd9, tdrg_icd10, acc_pdx, to_parallelize
          )
        })

        dt <- result$dt
        all_parts_summaries[[part]] <- result$consolidated_summary
        all_parts_statistics[[part]] <- result$aggregate_statistics

        # Print summary tables for individual parts
        # print_summary_tables(result, rows_to_show)
      } else {
        stop("Error: to_chunk must be TRUE; not chunking is deprecated.")
      }
    }
    if (to_write) {
      fwrite(dt, intermediate_file(part, fileext = TRUE))
    }

    if (to_group) {
      if (to_profvis) {
        p <- profvis({
          export_for_batch_grouper(dt, year_to_load, output_txt_file(part))
          for_batch_grouping <- fread(output_txt_file(part),
                                      sep = "|", na.strings = "--"
          )
          if (file.exists(grouper_result_file(part))) {
            batch_grouping_result <- fread(grouper_result_file(part),
                                           sep = "|", na.strings = "--"
            )
          }
        })
        htmlwidgets::saveWidget(
          p,
          file = here(
            "git-ignored-files", "profvis",
            paste0(
              "export_for_grouper_part_",
              paste0(part, "_of_", split_chunks), "_.html"
            )
          ),
          selfcontained = TRUE
        )
      } else {
        export_for_batch_grouper(dt, year_to_load, output_txt_file(part))
        for_batch_grouping <- fread(output_txt_file(part),
                                    sep = "|", na.strings = "--"
        )
        if (file.exists(grouper_result_file(part))) {
          batch_grouping_result <- fread(grouper_result_file(part),
                                         sep = "|", na.strings = "--"
          )
        }
      }
    }
  }

  # Combine the summaries and statistics from all parts
  final_combined_summary <- combine_all_parts_summaries(
    all_parts_summaries, rows_to_show
  )
  final_combined_statistics <- combine_all_parts_statistics(
    all_parts_statistics
  )

  # Print the final combined summary tables
  print_summary_tables(
    list(
      dt = NULL,
      consolidated_summary = final_combined_summary,
      aggregate_statistics = final_combined_statistics
    ),
    rows_to_show
  )
} else {
  part <- split_chunk_to_process
  print(paste("Processing Part", part, "only"))
}


## Read & Export Data (Per Part; Not Looping)

#### Split into N Parts & Read Data

In [ ]:
options(verbose = FALSE)
options(warn = -1)

if (!is.na(split_chunk_to_process)) {
  if (to_split) {
    rows_per_part <- ceiling(total_rows / split_chunks)
    for (part in 1:split_chunks) {
      chunk_file <- if (to_sample) {
        sampled_claims_file(part)
      } else {
        full_claims_file(part)
      }

      if (!file.exists(chunk_file)) {
        start_row <- (part - 1) * rows_per_part + 1
        end_row <- min(part * rows_per_part, total_rows)
        read_and_save_partial(start_row, end_row, part)
      } else {
        # print(paste("File already exists:", chunk_file))
      }
    }
  }

  part <- split_chunk_to_process
  chunk_file <- if (to_sample) {
    sampled_claims_file(part)
  } else {
    full_claims_file(part)
  }

  # print(paste("Reading chunk file:", chunk_file))

  if (!file.exists(chunk_file)) {
    print(paste("File does not exist. Creating file:", chunk_file))
    rows_per_part <- ceiling(total_rows / split_chunks)
    start_row <- (part - 1) * rows_per_part + 1
    end_row <- min(part * rows_per_part, total_rows)
    read_and_save_partial(start_row, end_row, part)
  } else {
    # Check file size
    file_info <- file.info(chunk_file)
    # print(paste("File size (bytes):", file_info$size))
  }

  dt <- main_read_function(file = chunk_file)
  # print(paste("Number of rows after reading:", nrow(dt))) # Add this line
}


In [ ]:
options(warn = 1)


### Data Cleaning

#### Chunking

In [ ]:
# Main code
if (!is.na(split_chunk_to_process)) {
  tic("Total execution time:")
  if (to_profvis) {
    p <- profvis({
      if (to_chunk) {
        num_cores <- max(1, availableCores() - 1)

        result <- parallelize_and_summarize(
          dt, num_cores,
          to_view_checks = to_view_checks_parallelized, global_seed,
          rows_to_show, rvs_icd9, tdrg_icd10, acc_pdx, to_parallelize
        )

        dt <- result$dt

        print_summary_tables(result, rows_to_show)
      } else {
        stop("Error: to_chunk must be TRUE; not chunking is deprecated.")
      }
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "parallelized.html"),
      selfcontained = TRUE
    )
  } else {
    if (to_chunk) {
      num_cores <- max(1, availableCores() - 1)

      result <- parallelize_and_summarize(
        dt, num_cores,
        to_view_checks = to_view_checks_parallelized, global_seed,
        rows_to_show, rvs_icd9, tdrg_icd10, acc_pdx, to_parallelize
      )

      dt <- result$dt

      print_summary_tables(result, rows_to_show)
    } else {
      stop("Error: to_chunk must be TRUE; not chunking is deprecated.")
    }
  }
}


#### Not Chunking (Deprecated)

##### Clean Data (Deprecated)

In [ ]:
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      clean_result <- clean_data(dt)
      dt <- clean_result$data
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "clean_data.html"),
      selfcontained = TRUE
    )
  } else {
    clean_result <- clean_data(dt)
    dt <- clean_result$data
  }
}


##### Map Codes (Deprecated)

In [ ]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      rvs_mapping_result <- map_rvs_icd9(dt$clin_rvs, rvs_icd9)
      dt[, icd9_list := rvs_mapping_result$icd9_list]
      map_then_compare_icd_mappings(
        tdrg_icd10,
        rows_to_show,
        invalid_rows_to_show = rows_to_show
      )
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "map_rvs.html"),
      selfcontained = TRUE
    )
  } else {
    rvs_mapping_result <- map_rvs_icd9(dt$clin_rvs, rvs_icd9)
    dt[, icd9_list := rvs_mapping_result$icd9_list]
    map_then_compare_icd_mappings(
      tdrg_icd10,
      rows_to_show,
      invalid_rows_to_show = rows_to_show
    )
  }
}


##### Find PDx (Deprecated)

In [ ]:
# Find PDX
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "find_pdx.html"),
      selfcontained = TRUE
    )
  } else {
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}


### Export

#### Export Intermediate Output

In [ ]:
if (!is.na(split_chunk_to_process)) {
  if (to_write) {
    fwrite(dt, intermediate_file(part, fileext = TRUE))
  }
}


#### Export for Batch Grouper

In [ ]:
if (!is.na(split_chunk_to_process)) {
  if (to_group) {
    if (to_profvis) {
      p <- profvis({
        export_for_batch_grouper(dt, year_to_load, output_txt_file(part))
        for_batch_grouping <- fread(output_txt_file(part),
          sep = "|", na.strings = "--"
        )
        if (file.exists(grouper_result_file(part))) {
          batch_grouping_result <- fread(grouper_result_file(part),
            sep = "|", na.strings = "--"
          )
        }
      })
      htmlwidgets::saveWidget(
        p,
        file = here("git-ignored-files", "profvis", "export_for_grouper.html"),
        selfcontained = TRUE
      )
    } else {
      export_for_batch_grouper(dt, year_to_load, output_txt_file(part))
      for_batch_grouping <- fread(output_txt_file(part),
        sep = "|", na.strings = "--"
      )
      if (file.exists(grouper_result_file(part))) {
        batch_grouping_result <- fread(grouper_result_file(part),
          sep = "|", na.strings = "--"
        )
      }
    }
  }
}


## Runtime Estimation

### Stop Timer

In [ ]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


### Calculate Speed

In [ ]:
if (!is.na(split_chunk_to_process)) {
  # Calculate time spent per cell and per row
  total_rows_dt <- nrow(dt)
  total_cells <- nrow(dt) * ncol(dt)

  time_per_cell <- total_time / total_cells
  time_per_row <- total_time / total_rows_dt
  time_estimate_total_rows <- time_per_row * total_rows

  # Format the row numbers
  formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
  formatted_total_rows <- format_large_numbers(total_rows)

  # Print the results with aligned decimal points and formatted row numbers
  cat(sprintf(
    "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
    formatted_total_rows_dt, total_time
  ))
  cat(sprintf(
    "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
    formatted_total_rows_dt, time_per_row * 1000
  ))
  cat(sprintf(
    "Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
    formatted_total_rows, time_estimate_total_rows / 60
  ))
} else {
  # Calculate time spent per cell and per row
  # total_rows_dt <- nrow(dt)
  total_cells <- nrow(dt) * ncol(dt)

  time_per_cell <- total_time / total_cells
  time_per_row <- total_time / total_rows
  # time_estimate_total_rows <- time_per_row * total_rows

  # Format the row numbers
  # formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
  formatted_total_rows <- format_large_numbers(total_rows)

  # Print the results with aligned decimal points and formatted row numbers
  cat(sprintf(
    "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
    formatted_total_rows, total_time
  ))
  cat(sprintf(
    "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
    formatted_total_rows, time_per_row * 1000
  ))
}


## Combine All Scripts (for debugging)

In [ ]:
# # List all .R files in the directory
# r_files <- list.files(
#   path = here("data-cleaning", "r_scripts"),
#   pattern = "\\.R$", full.names = TRUE
# )

# # Output file
# output_file <- "everything.R"

# # Read and concatenate contents
# file_contents <- lapply(r_files, readLines)
# concatenated_content <- unlist(file_contents)
# cat(concatenated_content,
#   file = paste0(here("data-cleaning", "everything", output_file)), sep = "\n"
# )
